In [1]:
import pandas as pd
import numpy as np

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [2]:
# Load the merged AID_654 + AID_1347034 dataset (already merged, not yet deduped)
hepg2_raw = pd.read_csv(
    '/Users/yamini/Desktop/projects/qsar casepase/qsar_prediction/Rebuttal/summary_2_1437034/Merging/merged_AID_654_1347034.csv',
    low_memory=False)

print("AID_654 + AID_1347034 (merged, HepG2)")
print(f"Loaded shape: {hepg2_raw.shape}")

# Remove rows with missing CID
hepg2_raw['PUBCHEM_CID'] = pd.to_numeric(hepg2_raw['PUBCHEM_CID'], errors='coerce')
print(f"Rows with missing CID: {hepg2_raw['PUBCHEM_CID'].isna().sum()}")
hepg2_raw = hepg2_raw.dropna(subset=['PUBCHEM_CID'])
hepg2_raw['PUBCHEM_CID'] = hepg2_raw['PUBCHEM_CID'].astype(int)
hepg2_raw['PUBCHEM_ACTIVITY_SCORE'] = pd.to_numeric(hepg2_raw['PUBCHEM_ACTIVITY_SCORE'], errors='coerce')

print(f"After cleanup: {hepg2_raw.shape}")
print(f"Unique CIDs: {hepg2_raw['PUBCHEM_CID'].nunique()}")

AID_654 + AID_1347034 (merged, HepG2)
Loaded shape: (11075, 44)
Rows with missing CID: 157
After cleanup: (10918, 44)
Unique CIDs: 7716


In [3]:
# Merge duplicates per CID:
# 1. Average PUBCHEM_ACTIVITY_SCORE across ALL measurements for that CID,
#    regardless of what outcome each measurement reported.
# 2. Classify the outcome from the averaged score itself: Inactive is always 0,
#    Inconclusive is 5-40, Active is 40-100 (checked directly on this dataset --
#    at score==40 the real outcome is Active in 41/45 cases, so >=40 -> Active
#    is the right cutoff here too, same rule as AID_720637). No 'Conflict'
#    category -- every compound gets a real outcome from its averaged score.

def outcome_from_score(score):
    if score == 0:
        return 'Inactive'
    elif score < 40:
        return 'Inconclusive'
    else:
        return 'Active'

agg_dict = {col: 'first' for col in hepg2_raw.columns
            if col not in ['PUBCHEM_CID', 'PUBCHEM_ACTIVITY_SCORE', 'PUBCHEM_ACTIVITY_OUTCOME']}
agg_dict['PUBCHEM_ACTIVITY_SCORE'] = 'mean'

hepg2_clean = hepg2_raw.groupby('PUBCHEM_CID', as_index=False).agg(agg_dict)
hepg2_clean['PUBCHEM_ACTIVITY_OUTCOME'] = hepg2_clean['PUBCHEM_ACTIVITY_SCORE'].apply(outcome_from_score)

print(f"After dedup: {hepg2_raw.shape} -> {hepg2_clean.shape}")
print(f"Removed {len(hepg2_raw) - len(hepg2_clean)} duplicate rows")
print()
print(hepg2_clean['PUBCHEM_ACTIVITY_OUTCOME'].value_counts())

After dedup: (10918, 44) -> (7716, 44)
Removed 3202 duplicate rows

PUBCHEM_ACTIVITY_OUTCOME
Inactive        7003
Inconclusive     431
Active           282
Name: count, dtype: int64


In [4]:
# Verify on a CID whose measurements actually disagreed on outcome -- pick the
# first such CID automatically rather than hardcoding one.
n_outcomes_per_cid = hepg2_raw.groupby('PUBCHEM_CID')['PUBCHEM_ACTIVITY_OUTCOME'].nunique()
disagreed_cids = n_outcomes_per_cid[n_outcomes_per_cid > 1].index
example_cid = disagreed_cids[0]

print(f"Verification - CID {example_cid} (measurements disagreed on outcome):")
print("\nBefore dedup (all rows for this CID):")
display(hepg2_raw[hepg2_raw['PUBCHEM_CID'] == example_cid][['PUBCHEM_CID', 'PUBCHEM_ACTIVITY_OUTCOME', 'PUBCHEM_ACTIVITY_SCORE']])

print("\nAfter dedup (one merged row -- score averaged, outcome from the averaged score):")
display(hepg2_clean[hepg2_clean['PUBCHEM_CID'] == example_cid][['PUBCHEM_CID', 'PUBCHEM_ACTIVITY_OUTCOME', 'PUBCHEM_ACTIVITY_SCORE']])

print(f"\nTotal CIDs whose measurements disagreed on outcome: {len(disagreed_cids)}")
print("All of these now get a real Active/Inactive/Inconclusive label from their")
print("averaged score -- no 'Conflict' category, nothing dropped.")

Verification - CID 177 (measurements disagreed on outcome):

Before dedup (all rows for this CID):


,PUBCHEM_CID,PUBCHEM_ACTIVITY_OUTCOME,PUBCHEM_ACTIVITY_SCORE
133,177,Inactive,0
1980,177,Inconclusive,30



After dedup (one merged row -- score averaged, outcome from the averaged score):


,PUBCHEM_CID,PUBCHEM_ACTIVITY_OUTCOME,PUBCHEM_ACTIVITY_SCORE
14,177,Inconclusive,15.0



Total CIDs whose measurements disagreed on outcome: 202
All of these now get a real Active/Inactive/Inconclusive label from their
averaged score -- no 'Conflict' category, nothing dropped.


In [5]:
# Save
hepg2_clean.to_csv('AID_654_1347034_cleaned.csv', index=False)
print(f"Saved {len(hepg2_clean)} compounds to AID_654_1347034_cleaned.csv")

Saved 7716 compounds to AID_654_1347034_cleaned.csv
